In [1]:
pip install transformers datasets torch pandas scikit-learn accelerate

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "GroNLP/hateBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True,
    id2label={0: "Non-Toxic", 1: "Insults and Flaming", 2: "Other Offensive Texts", 
              3: "Hate and Harassment 	", 4: "Threats", 5: "Extremism"},
    label2id={"Non-Toxic": 0, "Insults and Flaming": 1, "Other Offensive Texts": 2,
              "Hate and Harassment 	": 3, "Threats": 4, "Extremism": 5}
)

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10404.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider traini

In [10]:
from pathlib import Path

main_dir = Path.cwd()
print(main_dir)

c:\Users\dj140\OneDrive\Desktop\project\MSc-DSA-2026-Thesis-Project\Transformer_Model_Benchmarking\HateBERT


# Using GameTox

In [11]:
PATH_GAMETOX = "Existing_Datasets/GameTox/train.csv"

In [14]:
import pandas as pd

df = pd.read_csv( main_dir.parent / PATH_GAMETOX )

df.head()

,index,message,label
0,30702,no rush,0.0
1,18607,whatever ... watch the replay,0.0
2,32901,useless,1.0
3,25964,3 gunmark,0.0
4,28643,lol,0.0


In [15]:
df["label"] = df["label"].astype(int)
df = df[["message", "label"]]

In [16]:
df.tail()

,message,label
42954,давай,0
42955,i had over 7k combined,0
42956,like move your tank into the fight instead of ...,0
42957,ihr scheiss juden kinder,0
42958,omfg,2


In [17]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

dataset

DatasetDict({
    train: Dataset({
        features: ['message', 'label'],
        num_rows: 34367
    })
    test: Dataset({
        features: ['message', 'label'],
        num_rows: 8592
    })
})

In [23]:
def tokenize(batch):

    return tokenizer(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=200
    )

In [24]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 8592/8592 [00:00<00:00, 12418.89 examples/s]


In [ ]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./hateBERT-GameTox",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True
)

In [26]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy":
            accuracy_score(labels, predictions),

        "f1":
            f1_score(
                labels,
                predictions,
                average="macro"
            )
    }

In [27]:
from transformers import Trainer


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=
        tokenized_dataset["train"],

    eval_dataset=
        tokenized_dataset["test"],

    compute_metrics=
        compute_metrics
)


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.327275,0.290354,0.906075,0.368504
2,0.240961,0.325236,0.906774,0.406042
3,0.214774,0.350424,0.904213,0.457161


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]


TrainOutput(global_step=6444, training_loss=0.2657031042097815, metrics={'train_runtime': 2716.431, 'train_samples_per_second': 37.955, 'train_steps_per_second': 2.372, 'total_flos': 1.05968699877672e+16, 'train_loss': 0.2657031042097815, 'epoch': 3.0})

In [40]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

text = "you are garbage"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


with torch.no_grad():
    output = model(**inputs)


probabilities = torch.softmax(
    output.logits,
    dim=1
)

pred = torch.argmax(
    probabilities,
    dim=1
).item()

print(probabilities)
print(model.config.id2label[pred])

tensor([[0.0753, 0.8650, 0.0536, 0.0040, 0.0012, 0.0009]], device='cuda:0')
Insults and Flaming
